# 文書に答えがない質問を「答えなし」と判定する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/21_qa/qa_unanswerable.ipynb)

NLP クックブック 21-04 ／ 質問応答

抽出型 QA は、文書に答えがなくてもどこかを抜き出してしまう。答えがあるかどうかを判定して、ないときは「答えなし」と返したい。

In [1]:
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

name = "deepset/xlm-roberta-base-squad2"   # 答えのない質問を含む SQuAD 2.0 で学習した多言語モデル
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForQuestionAnswering.from_pretrained(name).eval()

@torch.no_grad()
def answer(question, context, max_len=30):
    enc = tokenizer(question, context, return_tensors="pt", truncation="only_second", max_length=512, return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0]
    out = model(**enc)
    start, end = out.start_logits[0], out.end_logits[0]
    null_score = (start[0] + end[0]).item()            # 先頭の特殊トークンを指す = 「答えなし」のスコア
    in_context = torch.tensor([s == 1 for s in enc.sequence_ids(0)])
    scores = start[:, None] + end[None, :]
    scores = scores.masked_fill(~in_context[:, None] | ~in_context[None, :], -1e9)
    scores = scores.triu().masked_fill(torch.ones_like(scores).tril(-1).bool(), -1e9).masked_fill(torch.ones_like(scores).triu(max_len).bool(), -1e9)
    s, e = divmod(scores.argmax().item(), scores.shape[1])
    return context[offsets[s][0]:offsets[e][1]], scores[s, e].item(), null_score

context = ("富士山は、静岡県と山梨県にまたがる活火山である。標高は 3776 メートルで、日本の最高峰である。"
           "2013 年に「富士山-信仰の対象と芸術の源泉」としてユネスコの世界文化遺産に登録された。")
questions = [("富士山の高さは？", True), ("富士山はどこにありますか？", True), ("世界遺産に登録されたのはいつ？", True),
             ("富士山に初めて登った人は誰ですか？", False), ("富士山の最後の噴火は何年ですか？", False), ("琵琶湖の面積は？", False)]
for q, answerable in questions:
    span, span_score, null_score = answer(q, context)
    diff = span_score - null_score                      # 差が小さい (または負) なら「答えなし」
    print(f"{'答えあり' if answerable else '答えなし'} | 差 {diff:6.1f} | {span[:24]!r:30} | {q}")

答えあり | 差   12.0 | '3776 メートル'                    | 富士山の高さは？
答えあり | 差    9.8 | '静岡県と山梨県'                      | 富士山はどこにありますか？
答えあり | 差    2.4 | '2013 年に'                      | 世界遺産に登録されたのはいつ？
答えなし | 差   -3.1 | '日本の最高峰'                       | 富士山に初めて登った人は誰ですか？
答えなし | 差   -6.4 | '2013'                         | 富士山の最後の噴火は何年ですか？
答えなし | 差   -5.9 | '3776 メートル'                    | 琵琶湖の面積は？


In [2]:
def answer_or_none(question, context, threshold=0.0):
    span, span_score, null_score = answer(question, context)
    return span if span_score - null_score > threshold else None

for q, _ in questions:
    print(answer_or_none(q, context), "←", q)

3776 メートル ← 富士山の高さは？
静岡県と山梨県 ← 富士山はどこにありますか？
2013 年に ← 世界遺産に登録されたのはいつ？
None ← 富士山に初めて登った人は誰ですか？
None ← 富士山の最後の噴火は何年ですか？
None ← 琵琶湖の面積は？


- SQuAD 2.0 形式で学習したモデルは、答えがないとき、先頭の特殊トークンを開始・終了位置として指すように学習されている。「最良の区間のスコア − 答えなしのスコア」が判定の材料になる。
- しきい値は、答えのある質問とない質問の両方を含む検証データで決める。高くすると「答えなし」が増え (誤答が減り、見逃しが増える)、低くするとその逆。
- 答えなしを学習していないモデル (SQuAD 1.1 や JSQuAD で学習したもの) では、この差は当てにならない。その場合は区間の確率にしきい値を設けるしかなく、精度は低い。
- 多言語モデルを英語のデータで学習したものなので、日本語専用のモデルより抽出の精度は落ちる。日本語で答えなしを含むデータを用意できるなら、それで追加学習するのが望ましい。